In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name_14b = "Qwen/Qwen2.5-14B-Instruct"

model_14b = AutoModelForCausalLM.from_pretrained(
    model_name_14b,
    dtype="auto",
    device_map="auto"
)
tokenizer_14b = AutoTokenizer.from_pretrained(model_name_14b)

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [2]:
import xml.etree.ElementTree as ET

dev_tree = ET.parse('./data/dev/archehr-qa.xml')
test_tree = ET.parse('./data/test/archehr-qa.xml')

dev_root = dev_tree.getroot()
test_root = test_tree.getroot()

In [3]:
dev_patient = [i.text.strip() for i in dev_root.iter('patient_narrative')]
dev_clinician = [i.text.strip() for i in dev_root.iter('clinician_question')]

test_patient = [i.text.strip() for i in test_root.iter('patient_narrative')]
test_clinician = [i.text.strip() for i in test_root.iter('clinician_question')]

print(len(dev_patient), len(dev_clinician), len(test_patient), len(test_clinician))

20 20 100 100


## Evaluations

In [4]:
import numpy as np
import torch
from evaluate import load
from enum import Enum

class BertScorer:
    def __init__(self, device):
        self.device = device
        self.bertscore = load("bertscore")

    def compute_scores(self, references, predictions):
        scores = self.bertscore.compute(
            references=references,
            predictions=predictions,
            model_type="distilbert-base-uncased",
            device=self.device,
            num_layers=4,
            batch_size=8,
            nthreads=4,
            all_layers=False,
            idf=False,
            lang="en",
            rescale_with_baseline=True,
            baseline_path=None,
        )
        return scores["f1"]

    def compute_overall_score(self, references, predictions):
        scores = self.compute_scores(references, predictions)
        return np.mean(scores)


class RougeType(Enum):
    ROUGE1 = "rouge1"
    ROUGE2 = "rouge2"
    ROUGEL = "rougeL"
    ROUGELSUM = "rougeLsum"


class RougeScorer:
    def __init__(self, rouges=["rouge1", "rouge2", "rougeL", "rougeLsum"]):
        self.rouge = load("rouge")
        self.rouge_types = [RougeType(rt) for rt in rouges]

    def compute_scores(self, references, predictions):
        scores = {rt.value: [] for rt in self.rouge_types}
        for r, p in zip(references, predictions):
            rouge_scores = self.rouge.compute(references=[r], predictions=[p])
            for rouge_type, rt_scores in scores.items():
                rt_scores.append(rouge_scores[rouge_type])
        return scores

    def compute_overall_score(self, references, predictions):
        scores = self.compute_scores(references, predictions)
        return {key: np.mean(value) for key, value in scores.items()}

device = "mps"

rouge_scorer = RougeScorer()
bert_scorer = BertScorer(device)

In [ ]:
# bert_score = bert_scorer.compute_overall_score(truth, initial_answers)
# rouge_score = rouge_scorer.compute_overall_score(truth, initial_answers)

## One prompt

### 4B

#### Prompt 1

In [18]:
# prepare the model input
SYSTEM_PROMPT = f"""
Your task to transform a free-text, patient-authored question into a clear and concise clinician-interpreted question that reflects how a clinician would query a smart electronic health record (EHR) system when preparing a response to the patient.
Your outputs should be very brief only focusing on the key question in less than 15 words.

Goal:
    Generate a single, well-formed clinician question that captures the core clinical information need implied by the patient’s narrative, phrased as a query to an intelligent EHR system.
Input:
    Patient‑authored question (Patient Question)
Expected output:
    Clinician‑interpreted question that captures the main medical concern(s) (≤ 15 words).

Examples:
    Patient Question: {dev_patient[0]}
    Clinician-interpreted Question: {dev_clinician[0]}

    Patient Question: {dev_patient[1]}
    Clinician-interpreted Question: {dev_clinician[1]}

    Patient Question: {dev_patient[2]}
    Clinician-interpreted Question: {dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question:{question}\nClinician-interpreted Question:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=30,
        temperature=0.01
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [19]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

#### Prompt 2

In [6]:
# prepare the model input
SYSTEM_PROMPT = f"""
You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question:{question}\nClinician-interpreted Question:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [7]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [8]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


0.3843627147376537 {'rouge1': np.float64(0.28541264875065714), 'rouge2': np.float64(0.12945063993977038), 'rougeL': np.float64(0.25633039766840604), 'rougeLsum': np.float64(0.25633039766840604)}


In [9]:
dev_answers

['Why was ERCP recommended over continuing medication-based treatment for CBD sludge?',
 'Why was he given lasix and his oxygen flow rate was reduced?',
 "What should I do about my son's ongoing irritation and headache after brain bleeding?",
 "Was the cardiac catheterization necessary given the patient's stable condition and normal cardiac enzymes?",
 'What could be causing my left upper quadrant chest pain following a trihexyphenidyl, thorazine, and cocaine overdose with prolonged QT?',
 'Did the doctors fail to diagnose the lung infection in time due to delayed culture testing and late report results?',
 'What happens to the vein surgery if Coumadin is stopped?',
 'Does poison in the bloodstream remain in the body long after exposure, and could it be affecting my writing abilities?',
 'What treatments were provided for the mother during her ICU stay after a complicated twin delivery with jaundice and bleeding, and could this have long-term health',
 'What is the likelihood that your

### 14B

#### Prompt 1

In [7]:
# prepare the model input
SYSTEM_PROMPT = f"""
Your task to transform a free-text, patient-authored question into a clear and concise clinician-interpreted question that reflects how a clinician would query a smart electronic health record (EHR) system when preparing a response to the patient.
Your outputs should be very brief only focusing on the key question in less than 15 words.

Goal:
    Generate a single, well-formed clinician question that captures the core clinical information need implied by the patient’s narrative, phrased as a query to an intelligent EHR system.
Input:
    Patient‑authored question (Patient Question)
Expected output:
    Clinician‑interpreted question that captures the main medical concern(s) (≤ 15 words).

Examples:
    Patient Question: {dev_patient[0]}
    Clinician-interpreted Question: {dev_clinician[0]}

    Patient Question: {dev_patient[1]}
    Clinician-interpreted Question: {dev_clinician[1]}

    Patient Question: {dev_patient[2]}
    Clinician-interpreted Question: {dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question:{question}\nClinician-interpreted Question:"},
    ]
    text = tokenizer_14b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_14b([text], return_tensors="pt").to(model_14b.device)
    
    # conduct text completion
    generated_ids = model_14b.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer_14b.decode(output_ids, skip_special_tokens=True)

    return content

In [8]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [10]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


0.39582759514451027 {'rouge1': np.float64(0.2785342995598263), 'rouge2': np.float64(0.10469718433105162), 'rougeL': np.float64(0.26230719333272), 'rougeLsum': np.float64(0.26230719333272)}


#### Prompt 2

In [11]:
# prepare the model input
SYSTEM_PROMPT = f"""
You are rewriting patient-authored questions into a single, concise clinician-interpreted question.

Task:
- Identify the main information need.
- Rewrite it as one short question a clinician would ask internally.
- Focus on intent, not medical detail.

Rules:
- Output ONE sentence only
- No explanations or background
- Do NOT add medical terminology
- Keep it as short and simple as possible (≤15 words)
- Prefer "Why", "What", "When", or "Did"

Examples:

Patient Question:
{dev_patient[0]}
Clinician-interpreted Question:
{dev_clinician[0]}

Patient Question:
{dev_patient[1]}
Clinician-interpreted Question:
{dev_clinician[1]}

Patient Question:
{dev_patient[2]}
Clinician-interpreted Question:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question:\n{question}\n\nClinician-interpreted Question:"},
    ]
    text = tokenizer_14b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_14b([text], return_tensors="pt").to(model_14b.device)
    
    # conduct text completion
    generated_ids = model_14b.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer_14b.decode(output_ids, skip_special_tokens=True)

    return content

In [13]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

In [14]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

0.39450974985957143 {'rouge1': np.float64(0.2747667494291645), 'rouge2': np.float64(0.09732247284878864), 'rougeL': np.float64(0.25672327116829496), 'rougeLsum': np.float64(0.25672327116829496)}


#### Prompt 3

In [19]:
# prepare the model input
SYSTEM_PROMPT = f"""
You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Input:\n{question}\n\nOutput:"},
    ]
    text = tokenizer_14b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_14b([text], return_tensors="pt").to(model_14b.device)
    
    # conduct text completion
    generated_ids = model_14b.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer_14b.decode(output_ids, skip_special_tokens=True)

    return content

In [20]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

In [21]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

0.4100103175267577 {'rouge1': np.float64(0.31137120510125604), 'rouge2': np.float64(0.12588216867934865), 'rougeL': np.float64(0.2948975354017799), 'rougeLsum': np.float64(0.2948975354017799)}


In [22]:
dev_answers

['Why was ERCP recommended over continued medication?',
 'Why was he given lasix and his oxygen flow rate was reduced?',
 "What should I do about my son's irritation and headache since he was released from the hospital?",
 'Was the cardiac catheterization necessary for me given my condition?',
 'What could be causing the chest pain in my left upper quadrant since the overdose?',
 'Was the lung infection diagnosed and treated及时呢？(Was the lung infection diagnosed and treated promptly?)\n\nNote: It seems there might be a slight mis',
 'What are the potential effects of stopping blood thinners on her previous vein surgery?',
 'Can toxins from the bloodstream persist in the body long-term?',
 'What long-term problems could result from the jaundice I experienced in the hospital?',
 'Will he be able to continue breathing on his own and survive given the circumstances?',
 'Is a setback in symptoms a normal part of viral meningitis recovery or a sign of relapse?',
 'What could be causing her st

## Multi Prompt

### 14B initial (prompt 3) + 4B hallucination detection

In [6]:
# prepare the model input
SYSTEM_PROMPT = f"""You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Input:\n{question}\n\nOutput:"},
    ]
    text = tokenizer_14b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_14b([text], return_tensors="pt").to(model_14b.device)
    
    # conduct text completion
    generated_ids = model_14b.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer_14b.decode(output_ids, skip_special_tokens=True)

    return content

In [7]:
# prepare the model input
CLEANING_PROMPT = f"""You are tasked with detecting and removing hallucinations from an LLM output.
You will be given outputs from an LLM which should just contain a concise question.
If the output fits this criteria, simply return the same output.
If the output contains explanations, unnecessary content, hallucinations, etc. remove them and only return the actual question.
Do NOT explain or summarize
Do NOT add new details
"""

def get_clear_answer(question):
    messages = [
        {"role": "system", "content": CLEANING_PROMPT},
        {"role": "user", "content": f"LLM Output:\n{question}\n\nCleared Output:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [8]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [9]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


0.3837074749171734 {'rouge1': np.float64(0.30993198531720867), 'rouge2': np.float64(0.12588216867934865), 'rougeL': np.float64(0.28667670642233023), 'rougeLsum': np.float64(0.28667670642233023)}


In [12]:
clear_answers = []
for q in dev_answers:
    a = get_clear_answer(q)
    clear_answers.append(a)

In [13]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, clear_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, clear_answers)

print(bert_score, rouge_score)

0.41940785348415377 {'rouge1': np.float64(0.31064968866649095), 'rouge2': np.float64(0.12588216867934865), 'rougeL': np.float64(0.2873944097716125), 'rougeLsum': np.float64(0.2873944097716125)}


In [14]:
clear_answers

['Why was ERCP recommended over continued medication?',
 'Why was he given lasix and his oxygen flow rate was reduced?',
 "What should I do about my son's irritation and headache since he was released from the hospital?",
 'Was the cardiac catheterization necessary for me given my condition?',
 'What could be causing the chest pain in my left upper quadrant since the overdose?',
 'Why was the lung infection diagnosed and treated?',
 'What are the potential effects of stopping blood thinners on her previous vein surgery?',
 'Can toxins from the bloodstream persist in the body long-term?',
 'What long-term problems could result from the jaundice I experienced in the hospital?',
 'Will he be able to continue breathing on his own and survive given the circumstances?',
 'Is a setback in symptoms during recovery from viral meningitis a normal part of the process or a sign of relapse?',
 'What could be causing her stomach pain that extends to her chest and back?',
 'What can I do to relieve m

In [15]:
test_answers = []
for q in test_patient:
    a = get_answer(q)
    test_answers.append(a)

clear_test_answers = []
for q in test_answers:
    a = get_clear_answer(q)
    clear_test_answers.append(a)

In [16]:
bert_score = bert_scorer.compute_overall_score(test_clinician, test_answers)
rouge_score = rouge_scorer.compute_overall_score(test_clinician, test_answers)

print(bert_score, rouge_score)

bert_score = bert_scorer.compute_overall_score(test_clinician, clear_test_answers)
rouge_score = rouge_scorer.compute_overall_score(test_clinician, clear_test_answers)

print(bert_score, rouge_score)

0.34550071209669114 {'rouge1': np.float64(0.22855740062867724), 'rouge2': np.float64(0.04575583437723148), 'rougeL': np.float64(0.19742719788581164), 'rougeLsum': np.float64(0.19801543317992934)}
0.3476670618355274 {'rouge1': np.float64(0.22917885217975506), 'rouge2': np.float64(0.04575583437723148), 'rougeL': np.float64(0.1991774252715477), 'rougeLsum': np.float64(0.1991774252715477)}


## LoRA

### 4B

#### Prompt 1

In [23]:
SYSTEM_PROMPT = """You are a clinical assistant.
Your task to transform a free-text, patient-authored question into a clear and concise clinician-interpreted question that reflects how a clinician would query a smart electronic health record (EHR) system when preparing a response to the patient.

Goal:
    Generate a single, well-formed clinician question that captures the core clinical information need implied by the patient’s narrative, phrased as a query to an intelligent EHR system.
Input:
    Patient‑authored question (Patient Question)
Expected output:
    Clinician‑interpreted question that captures the main medical concern(s) (≤ 15 words).
"""

In [24]:
chats = []
for qp, qc in zip(test_patient, test_clinician):
    chat = {
      "messages": [
        {
          "role": "system",
          "content": SYSTEM_PROMPT
        },
        {
          "role": "user",
          "content": f"Patient Question: {qp}"
        },
        {
          "role": "assistant",
          "content": qc
        }
      ]
    }
    chats.append(chat)

print(len(chats))

100


In [25]:
chats[0]

{'messages': [{'role': 'system',
   'content': 'You are a clinical assistant.\nYour task to transform a free-text, patient-authored question into a clear and concise clinician-interpreted question that reflects how a clinician would query a smart electronic health record (EHR) system when preparing a response to the patient.\n\nGoal:\n    Generate a single, well-formed clinician question that captures the core clinical information need implied by the patient’s narrative, phrased as a query to an intelligent EHR system.\nInput:\n    Patient‑authored question (Patient Question)\nExpected output:\n    Clinician‑interpreted question that captures the main medical concern(s) (≤ 15 words).\n'},
  {'role': 'user',
   'content': 'Patient Question: My partner has end stage liver disease (alcoholic cirrhosis). Hepatic encephalopathy seems advanced. He is home, but because of his confusion and disorientation, he cant be left alone. He has been in and out of the hospital/ICU several times. His doc

In [31]:
import json

with open("train_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in chats:
        f.write(json.dumps(item) + "\n")

In [35]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="train_dataset.jsonl",
    split="train"
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 100
})

In [28]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,                     # small rank to avoid overfitting
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 2,949,120 || all params: 4,025,417,216 || trainable%: 0.0733


In [30]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)

In [42]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

trainer.train()

/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.697400
20,3.321700
30,3.176700


/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=39, training_loss=3.324145243718074, metrics={'train_runtime': 460.1622, 'train_samples_per_second': 0.652, 'train_steps_per_second': 0.085, 'total_flos': 1789117020785664.0, 'train_loss': 3.324145243718074})

In [45]:
def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question: {question}\nClinician-interpreted question: "},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

#### Prompt 2

In [5]:
# prepare the model input
SYSTEM_PROMPT = f"""You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

In [6]:
chats = []
for qp, qc in zip(test_patient, test_clinician):
    chat = {
      "messages": [
        {
          "role": "system",
          "content": SYSTEM_PROMPT
        },
        {
          "role": "user",
          "content": f"Input:\n{qp}\n\nOutput:"
        },
        {
          "role": "assistant",
          "content": qc
        }
      ]
    }
    chats.append(chat)

print(len(chats))

100


In [7]:
chats[0]

{'messages': [{'role': 'system',
   'content': 'You are converting long, messy questions into their single core question.\n\nTask:\n- Identify what the person is really asking.\n- Rewrite it as ONE short, clear question.\n- Remove extra details, emotion, and background.\n\nRules:\n- One sentence only\n- Keep it as short as possible\n- Do NOT explain or summarize\n- Do NOT add new details\n- Prefer simple "Why / What / Did" questions\n- Output ONLY the final question. No other text.\n\nExamples:\n\nInput:\nI had severe abdomen pain and was hospitalised for 15 days in ICU, diagnoised with CBD sludge thereafter on udiliv. Doctor advised for ERCP. My question is if the sludge was there does not the medication help in flushing it out? Whether ERCP was the only cure?\nOutput:\nWhy was ERCP recommended to him over continuing a medication-based treatment?\n\nInput:\nI just wrote about my dad given multiple shots of lasciks after he was already so swelled his shin looked like it would burst ope

In [8]:
import json

with open("train_dataset_prompt2.jsonl", "w", encoding="utf-8") as f:
    for item in chats:
        f.write(json.dumps(item) + "\n")

In [6]:
import json

chats = []
for qp, qc in zip(dev_patient[3:], dev_clinician[3:]):
    chat = {
      "messages": [
        {
          "role": "system",
          "content": SYSTEM_PROMPT
        },
        {
          "role": "user",
          "content": f"Input:\n{qp}\n\nOutput:"
        },
        {
          "role": "assistant",
          "content": qc
        }
      ]
    }
    chats.append(chat)

print(len(chats))

with open("dev_dataset_prompt2.jsonl", "w", encoding="utf-8") as f:
    for item in chats:
        f.write(json.dumps(item) + "\n")

17


In [10]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="train_dataset_prompt2.jsonl",
    split="train"
)

dataset

Dataset({
    features: ['messages'],
    num_rows: 100
})

In [9]:
dev_dataset = load_dataset(
    "json",
    data_files="dev_dataset_prompt2.jsonl",
    split="train"
)

dev_dataset

Dataset({
    features: ['messages'],
    num_rows: 17
})

In [11]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,                     # small rank to avoid overfitting
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 2,949,120 || all params: 4,025,417,216 || trainable%: 0.0733


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    num_train_epochs=5,              # allow more epochs
    fp16=True,
    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none"
)


In [16]:
from transformers import EarlyStoppingCallback
from trl import SFTTrainer

trainer = SFTTrainer(
    model=lora_model,
    train_dataset=dataset,
    eval_dataset=dev_dataset,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()


/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Tokenizing eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.786000,0.864713
2,0.701100,0.739191
3,0.725200,0.737608
4,0.589900,0.744784


/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=100, training_loss=1.0575941944122313, metrics={'train_runtime': 5461.2466, 'train_samples_per_second': 0.092, 'train_steps_per_second': 0.023, 'total_flos': 4247066504835072.0, 'train_loss': 1.0575941944122313})

In [17]:
def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Input:\n{question}\n\nOutput:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(lora_model.device)
    
    # conduct text completion
    generated_ids = lora_model.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [18]:
bert_score = bert_scorer.compute_overall_score(dev_clinician[3:], dev_answers[3:])
rouge_score = rouge_scorer.compute_overall_score(dev_clinician[3:], dev_answers[3:])

/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


In [19]:
print(bert_score)
print(rouge_score)

0.32140236654702353
{'rouge1': np.float64(0.20231021391647874), 'rouge2': np.float64(0.08557460417322008), 'rougeL': np.float64(0.18578195555431032), 'rougeLsum': np.float64(0.18578195555431032)}


In [20]:
dev_answers

['Why was ERCP recommended to him over continuing a medication-based treatment?',
 'Why was he given lasix and his oxygen flow rate was reduced?',
 'What is the expected course of recovery for him?',
 'Why was cardiac catheterization performed?',
 'What is the expected course of recovery for him?',
 'Why was the patient admitted to ICU and what were the reasons for his death?',
 'Was she on Coumadin after the surgery?',
 'Did she have any post-illness issues?',
 'What was the treatment for her jaundice?',
 'What is the expected course of recovery for him?',
 'What is the expected course of recovery for her?',
 'What is the expected course of recovery for her?',
 'What is the expected course of recovery for him?',
 'What is the diagnosis?',
 'What is the expected course of recovery for him?',
 'Is there a possibility of a stroke?',
 'What is the expected course of recovery for her?',
 'Is she expected to recover from the accident?',
 'Is there a possibility that the patient has a heart 

### 14B

In [5]:
SYSTEM_PROMPT = f"""
You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

In [27]:
chats = []
for qp, qc in zip(test_patient, test_clinician):
    chat = {
      "messages": [
        {
          "role": "system",
          "content": SYSTEM_PROMPT
        },
        {
          "role": "user",
          "content": f"Input:\n{qp}\n\nOutput:"
        },
        {
          "role": "assistant",
          "content": qc
        }
      ]
    }
    chats.append(chat)

print(len(chats))

100


In [28]:
chats[0]

{'messages': [{'role': 'system',
   'content': '\nYou are converting long, messy questions into their single core question.\n\nTask:\n- Identify what the person is really asking.\n- Rewrite it as ONE short, clear question.\n- Remove extra details, emotion, and background.\n\nRules:\n- One sentence only\n- Keep it as short as possible\n- Do NOT explain or summarize\n- Do NOT add new details\n- Prefer simple "Why / What / Did" questions\n- Output ONLY the final question. No other text.\n\nExamples:\n\nInput:\nI had severe abdomen pain and was hospitalised for 15 days in ICU, diagnoised with CBD sludge thereafter on udiliv. Doctor advised for ERCP. My question is if the sludge was there does not the medication help in flushing it out? Whether ERCP was the only cure?\nOutput:\nWhy was ERCP recommended to him over continuing a medication-based treatment?\n\nInput:\nI just wrote about my dad given multiple shots of lasciks after he was already so swelled his shin looked like it would burst o

In [29]:
import json

with open("train_dataset_14b.jsonl", "w", encoding="utf-8") as f:
    for item in chats:
        f.write(json.dumps(item) + "\n")

In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="train_dataset_14b.jsonl",
    split="train"
)

dataset

Dataset({
    features: ['messages'],
    num_rows: 100
})

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,                     # small rank to avoid overfitting
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model_14b = get_peft_model(model_14b, lora_config)
model_14b.print_trainable_parameters()

trainable params: 6,291,456 || all params: 14,776,325,120 || trainable%: 0.0426


In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-3,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model_14b,
    train_dataset=dataset,
    args=training_args,
)

trainer.train()